# 05b — Telugu ASR: MSCIL-Only Baseline (EXP-1)

Identical to `05_improved_full_training.ipynb` but **MSCIL data only**  
(no Shrutilipi, no OpenSLR-66). Purpose: establish how much KenLM helps  
before moving to the multi-dataset learning curve study in notebook 07.

| Setting | Value |
|---|---|
| Model | facebook/wav2vec2-xls-r-300m |
| Dataset | `./telugu_asr_clean_dataset/` (MSCIL ~35k clips, ~90h) |
| Processor | `./telugu_wav2vec2_processor/` |
| SpecAugment mask_feature_prob | **0.1** (key EXP-1 fix) |
| Decoding | Greedy → KenLM 5-gram |

In [1]:
!pip install -q \
    "typing_extensions>=4.6" \
    "transformers>=4.46" \
    "datasets<3.0" \
    "accelerate>=0.26" \
    "evaluate" "jiwer" "soundfile" "librosa" "pyctcdecode"
!pip install -q kenlm 2>/dev/null || echo "kenlm skipped"
print("Done.")

Done.


In [2]:
import logging, os, re, json, subprocess, sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Union

import numpy as np
import torch
import evaluate
from datasets import load_from_disk, Audio
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from huggingface_hub import login

logging.getLogger("numba").setLevel(logging.WARNING)

_hf_token = os.environ.get("HF_TOKEN")
if _hf_token:
    login(token=_hf_token)

print("Imports OK")

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [3]:
# ── Paths — MSCIL only ────────────────────────────────────────────────────
DATASET_PATH   = "./telugu_asr_clean_dataset"
PROCESSOR_PATH = "./telugu_wav2vec2_processor"
MODEL_NAME     = "facebook/wav2vec2-xls-r-300m"
OUTPUT_DIR     = "./wav2vec2-telugu-mscil-only"
VOCAB_JSON     = "./vocab.json"

assert Path(DATASET_PATH).exists(),   f"Not found: {DATASET_PATH}"
assert Path(PROCESSOR_PATH).exists(), f"Not found: {PROCESSOR_PATH}"

# ── EXP-1 hyperparameters ─────────────────────────────────────────────────
LR                   = 1e-4
WARMUP               = 1000
EPOCHS               = 50
EVAL_STEPS           = 500
SAVE_STEPS           = 500
SPLIT_RATIO          = 0.1
SEED                 = 42
FP16                 = True
MASK_TIME_PROB       = 0.1
MASK_FEAT_PROB       = 0.1   # key fix — was 0.004 in baseline
ATTENTION_DROPOUT    = 0.1
HIDDEN_DROPOUT       = 0.1
FEAT_PROJ_DROPOUT    = 0.0
EARLY_STOP_PATIENCE  = 8
EARLY_STOP_THRESHOLD = 0.005

# ── Memory-adaptive batch sizing ──────────────────────────────────────────
assert torch.cuda.is_available(), "No CUDA device found."
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb >= 40:    BATCH_SIZE, GRAD_ACCUM = 8, 4
elif vram_gb >= 20:  BATCH_SIZE, GRAD_ACCUM = 4, 8
elif vram_gb >= 12:  BATCH_SIZE, GRAD_ACCUM = 2, 16
else:                BATCH_SIZE, GRAD_ACCUM = 1, 32

torch.backends.cuda.matmul.allow_tf32 = True

print(f"GPU      : {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB)")
print(f"Dataset  : {DATASET_PATH}")
print(f"Batch    : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"mask_feature_prob : {MASK_FEAT_PROB}  (must be 0.1)")

GPU      : NVIDIA RTX A6000  (51 GB)
Dataset  : ./telugu_asr_clean_dataset
Batch    : 8 × 4 = 32 effective
mask_feature_prob : 0.1  (must be 0.1)


In [4]:
processor = Wav2Vec2Processor.from_pretrained(PROCESSOR_PATH)
print(f"Vocab size : {processor.tokenizer.vocab_size}")

dataset = load_from_disk(DATASET_PATH)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
print(f"Total samples : {len(dataset):,}")
print(f"Columns       : {dataset.column_names}")

split    = dataset.train_test_split(test_size=SPLIT_RATIO, seed=SEED)
train_ds = split["train"]
eval_ds  = split["test"]
print(f"Train : {len(train_ds):,}")
print(f"Eval  : {len(eval_ds):,}")

Vocab size : 67
Total samples : 35,426
Columns       : ['audio', 'transcription', 'audio_id']
Train : 31,883
Eval  : 3,543


In [5]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

KEEP = {"input_values", "input_length", "labels"}

print("Preparing train...")
train_prep = train_ds.map(
    prepare_dataset,
    remove_columns=[c for c in train_ds.column_names if c not in KEEP],
    num_proc=4, desc="train",
).sort("input_length")

print("Preparing eval...")
eval_prep = eval_ds.map(
    prepare_dataset,
    remove_columns=[c for c in eval_ds.column_names if c not in KEEP],
    num_proc=4, desc="eval",
)

print(f"Train prepared : {len(train_prep):,}")
print(f"Eval prepared  : {len(eval_prep):,}")

Preparing train...
Preparing eval...
Train prepared : 31,883
Eval prepared  : 3,543


In [6]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, padding=self.padding, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt"
        )
        batch["labels"] = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids  = np.argmax(pred.predictions, axis=-1)
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}

print("Collator and metrics ready.")

Collator and metrics ready.


In [7]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=ATTENTION_DROPOUT,
    hidden_dropout=HIDDEN_DROPOUT,
    feat_proj_dropout=FEAT_PROJ_DROPOUT,
    mask_time_prob=MASK_TIME_PROB,
    mask_time_length=10,
    mask_feature_prob=MASK_FEAT_PROB,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=processor.tokenizer.vocab_size,
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params  : {trainable:,}")
print(f"mask_feature_prob : {model.config.mask_feature_prob}")

Loading weights: 100%|██████████| 422/422 [00:00<00:00, 10750.58it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params  : 311,297,219
mask_feature_prob : 0.1


In [8]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP,
    lr_scheduler_type="cosine",
    fp16=FP16,
    dataloader_num_workers=4,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=100,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
    label_names=["labels"],
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_prep,
    eval_dataset=eval_prep,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOP_PATIENCE,
            early_stopping_threshold=EARLY_STOP_THRESHOLD,
        )
    ],
)

print(f"Starting training on MSCIL-only ({len(train_prep):,} samples)...")
trainer.train()

FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"Model saved to {FINAL_DIR}")

Starting training on MSCIL-only (31,883 samples)...


Step,Training Loss,Validation Loss,Wer,Cer
500,14.124343,3.497806,1.000000,0.983584
1000,5.803679,0.951301,0.858145,0.248037
1500,4.186099,0.626586,0.716501,0.181493
2000,3.605707,0.531454,0.656455,0.159929
2500,3.250782,0.479846,0.614705,0.147490
3000,3.087008,0.451176,0.593323,0.139844
3500,2.904969,0.433636,0.573970,0.134308
4000,2.737120,0.415804,0.559603,0.128046
4500,2.558804,0.396452,0.548489,0.124660
5000,2.523002,0.390065,0.535221,0.120558


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.23s/it]


Model saved to ./wav2vec2-telugu-mscil-only/final


In [9]:
# ── Part A: Greedy WER ────────────────────────────────────────────────────
final_metrics = trainer.evaluate()
greedy_wer = final_metrics.get("eval_wer", float("nan"))
greedy_cer = final_metrics.get("eval_cer", float("nan"))

print("=" * 55)
print("GREEDY DECODING")
print(f"  WER : {greedy_wer:.4f}  ({greedy_wer*100:.2f}%)")
print(f"  CER : {greedy_cer:.4f}  ({greedy_cer*100:.2f}%)")
print("=" * 55)

# ── Part B: KenLM 5-gram ──────────────────────────────────────────────────
KENLM_BIN = "./kenlm/build/bin/lmplz"
ARPA_PATH = "./telugu_kenlm_mscil.arpa"
LM_TEXT   = "./telugu_lm_mscil.txt"

kenlm_wer = float("nan")

if Path(KENLM_BIN).exists():
    import pyctcdecode

    # Build LM from train transcripts
    with open(LM_TEXT, "w", encoding="utf-8") as f:
        f.write("\n".join(train_ds["transcription"]))
    print(f"Wrote {len(train_ds):,} transcripts → {LM_TEXT}")

    print("Building 5-gram ARPA...")
    with open(LM_TEXT, "rb") as fin, open(ARPA_PATH, "w") as fout:
        subprocess.run(
            [KENLM_BIN, "-o", "5", "--discount_fallback"],
            stdin=fin, stdout=fout, check=True,
        )
    print(f"ARPA saved → {ARPA_PATH}")

    with open(VOCAB_JSON) as f:
        vocab_dict = json.load(f)
    vocab_list = sorted(vocab_dict.keys(), key=lambda k: vocab_dict[k])

    decoder = pyctcdecode.build_ctcdecoder(
        vocab_list, ARPA_PATH, alpha=0.5, beta=1.0
    )
    print("KenLM decoder ready.")

    # Decode eval set
    inference_model  = trainer.model.eval()
    inference_device = next(inference_model.parameters()).device
    preds, refs = [], []

    print(f"Decoding {len(eval_ds):,} samples with KenLM...")
    for i, sample in enumerate(eval_ds):
        inp = processor(
            sample["audio"]["array"], sampling_rate=16_000,
            return_tensors="pt", padding=True,
        )
        with torch.no_grad():
            logits = inference_model(
                input_values=inp.input_values.to(inference_device)
            ).logits[0].cpu().numpy()
        preds.append(decoder.decode(logits))
        refs.append(sample["transcription"])
        if (i + 1) % 500 == 0:
            print(f"  {i+1}/{len(eval_ds)} done")

    kenlm_wer = wer_metric.compute(predictions=preds, references=refs)

    print()
    print("=" * 55)
    print("MSCIL-ONLY RESULTS")
    print(f"  Greedy WER : {greedy_wer:.4f}  ({greedy_wer*100:.2f}%)")
    print(f"  KenLM  WER : {kenlm_wer:.4f}  ({kenlm_wer*100:.2f}%)")
    print(f"  KenLM improvement : {(greedy_wer - kenlm_wer)*100:.1f}% absolute")
    print("=" * 55)

    results = {
        "dataset": "mscil_only",
        "n_train": len(train_ds),
        "n_eval":  len(eval_ds),
        "greedy_wer": round(greedy_wer, 4),
        "greedy_cer": round(greedy_cer, 4),
        "kenlm_wer":  round(kenlm_wer, 4),
        "kenlm_improvement_abs": round(greedy_wer - kenlm_wer, 4),
    }
    Path("./results").mkdir(exist_ok=True)
    with open("./results/wav2vec2_mscil_only.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Results saved → ./results/wav2vec2_mscil_only.json")

else:
    print(f"[WARN] KenLM binary not found at {KENLM_BIN}")
    print("Build: cd kenlm && mkdir build && cd build && cmake .. && make -j4")
    print(f"Greedy WER : {greedy_wer:.4f}")

Training Loss,Validation Loss,Step,Wer,Cer
1.661905,0.326359,15500,0.462751,0.100716


GREEDY DECODING
  WER : 0.4628  (46.28%)
  CER : 0.1007  (10.07%)
Wrote 31,883 transcripts → ./telugu_lm_mscil.txt
Building 5-gram ARPA...


=== 1/5 Counting and sorting n-grams ===
Reading /home/jovyan/work/asr_lrl_mscil_ftl/telugu_lm_mscil.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 209357 types 40048
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:480576 2:10518434816 3:19722065920 4:31555305472 5:46018154496
Statistics:
1 40048 D1=0.726056 D2=1.07822 D3+=1.27928
2 144520 D1=0.864599 D2=1.26604 D3+=1.40796
3 176461 D1=0.936107 D2=1.41995 D3+=1.55085
4 161585 D1=0.962421 D2=1.70884 D3+=1.68789
5 136389 D1=0.869859 D2=1.82564 D3+=2.76289
Memory estimate for binary LM:
type       kB
probing 14724 assuming -p 1.5
probing 17708 assuming -r models -p 1.5
trie     7435 without quantization
trie     4290 assuming -q 8 -b 8 quantization 
trie     6781 assuming -a 22 array pointer compression
trie     3636 assuming -a 22 -q 8 -b 8 array p

ARPA saved → ./telugu_kenlm_mscil.arpa


****************************************************************************************************


KenLM decoder ready.
Decoding 3,543 samples with KenLM...
  500/3543 done
  1000/3543 done
  1500/3543 done
  2000/3543 done
  2500/3543 done
  3000/3543 done
  3500/3543 done

MSCIL-ONLY RESULTS
  Greedy WER : 0.4628  (46.28%)
  KenLM  WER : 0.2844  (28.44%)
  KenLM improvement : 17.8% absolute
Results saved → ./results/wav2vec2_mscil_only.json
